# Kinthread v10-r9p on Kaggle (Save & Run All)
r8p warm start (checked by SHA-256), r8p data plus `data/balanced_mix_v1` (every question format at one count, detours, lines of descent up to ten generations, two-family stories), block 512 for 192-token readings. Picked and stopped by real dev questions; also scores the gauntlet and the repeatability suite. Never scores the locked test (see TRAINING_V11_CODEX.md step 6). Outputs in /kaggle/working/runs.

In [ ]:
import os, glob, hashlib, subprocess
W = '/kaggle/working'
REPO = '/kaggle/temp/Kinthread-llm-model'
BRANCH = 'claude/nice-hamilton-3ttcwr'   # until it is merged into v9-reader
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    !git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}
%cd {REPO}
R8P = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'   # v10-r8p step 6,000, the FINAL pick
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
cands = glob.glob('/kaggle/input/**/best.pt', recursive=True)
try:   # the r8p notebook output (versions 7 and 5), when it is not attached
    import kagglehub
    for h in ('anirudharavalli/notebook99ae0586bd/versions/7', 'anirudharavalli/notebook99ae0586bd/versions/5'):
        try:
            cands += glob.glob(f'{kagglehub.notebook_output_download(h)}/**/best.pt', recursive=True)
        except Exception as e:
            print(h, e)
except Exception as e:
    print('kagglehub:', e)
INIT = next((p for p in cands if sha(p) == R8P), None)
assert INIT, f'attach the v10-r8p notebook output (best.pt with SHA-256 {R8P}); found {cands}'
!cd data && sha256sum -c SHA256SUMS --ignore-missing --quiet && echo data ok
!pip -q install -U gdown
S = '/kaggle/temp/share'
os.makedirs(S, exist_ok=True)
if not os.path.isdir(f'{S}/dataset_v7'):
    !gdown -q --folder https://drive.google.com/drive/folders/1MNiQuramSX5ImytxEdvNEIN7SgFtT6k8 -O {S}/dataset_v7
FILES = {'real_questions.tgz': '1yIanNsRCb6-BAkGIF_SIjp9ZMCRIotGy', 'star_r3.jsonl.gz': '1_qk0JL_lkzKBdv2vwHxWijI4yNrNPRtr',
         'star_r2.jsonl.gz': '1s_RQLHpNbWsttRTjaoFpV1NP5I4q3Veh', 'corrections.jsonl.gz': '1VW_dWGo2GkDdV9AKVWl-pxz-IFPrpEZp'}
for n, fid in FILES.items():
    if not os.path.exists(f'{S}/{n}'):
        !wget -q -O {S}/{n} "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"
os.makedirs('data/real_questions', exist_ok=True)
!tar -xzf {S}/real_questions.tgz -C data/real_questions 2>/dev/null
DATA = f'{S}/dataset_v7'
OUT = f'{W}/runs/v10-r9p'
os.makedirs(OUT, exist_ok=True)
print('data', os.listdir(DATA), '| real', len(os.listdir('data/real_questions')))tgz = glob.glob('/kaggle/input/**/real_questions_full.tgz', recursive=True)
if tgz:   # the private Kaggle dataset, when attached, replaces the Drive copy
    !rm -rf data/real_questions && tar -xzf {tgz[0]}
print('init', INIT, '| real', len(glob.glob('data/real_questions/*.jsonl')))


In [ ]:
EXTRA = [f'{DATA}/reworded.jsonl.gz*2', 'data/mix_r8.jsonl.gz', f'{S}/star_r3.jsonl.gz*2',
         'data/balanced_mix_v1.jsonl.gz*2']   # + every format at one count, detours, ten-generation lines, two-family stories
EXTRA = ','.join(e for e in EXTRA if os.path.exists(e.split('*')[0]))
print(EXTRA)
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_reader.py --arch v10 --run v10-r9p --out {OUT} --pool {DATA} --init {INIT} --prefix_lm --of_form --style_p 0.8 --steps 15000 --batch 64 --lr 3e-4 --warmup 500 --optim muon --amp --workers 2 --resume --eval_every 1500 --save_every 1000 --reworded {EXTRA} --reworded_dev data/reworded_dev.jsonl.gz --p_reworded 0.75 --ramp_steps 2000 --block 512 --real_dev 575 --select_real


In [ ]:
!python -m eval.run_real --ckpt {OUT}/best.pt --split dev --adaptive 1.0 --hybrid --out {OUT}/real_dev.json | grep -E '"dev'
!python -m eval.run_real --ckpt {OUT}/best.pt --split dev --adaptive 1.0 --hybrid --max_new 192 --reserve 8 --out {OUT}/real_dev_192.json | grep -E '"dev'
!python -m eval.gauntlet --ckpt {OUT}/best.pt --k 16 --out {OUT}/gauntlet.json | head -5
!python -m eval.consistency --groups 300 --ckpt {OUT}/best.pt --k 16 --out {OUT}/consistency.json | head -4
!python -m eval.construction_suite --ckpt {OUT}/best.pt --n 15 --out {OUT}/construction_suite.json | tail -3
!rm -f {OUT}/latest.pt
